In [ ]:
import pandas as pd

In [ ]:
import io

In [ ]:
from re import U
from google.colab import files
uploaded = files.upload()

Saving smit_data_1 - Sheet1.csv to smit_data_1 - Sheet1.csv


In [ ]:
import pandas as pd
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

# Load Pegasus model and tokenizer
model_name = 'google/pegasus-large'
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.09k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [ ]:

# Function to generate summary using Pegasus
def generate_summary(text):
    inputs = tokenizer([text], max_length=1024, return_tensors='pt', truncation=True)
    summary_ids = model.generate(inputs['input_ids'], max_length=150, num_beams=4, length_penalty=2.0, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Read CSV file
df = pd.read_csv('smit_data_1 - Sheet1.csv')

# Apply summarization on 'News Content in Detail' column and create 'News Content Summary' column
df['News Content Summary'] = df['News Content in Detail'].apply(generate_summary)

# Print the News Content in Detail and the generated News Content Summary
for index, row in df.iterrows():
    print(f"Original Content:\n{row['News Content in Detail']}\n")
    print(f"Generated Summary:\n{row['News Content Summary']}\n")
    print("---------------------------------------------------\n")

# Save updated DataFrame back to CSV
df.to_csv('summarized_file.csv', index=False)


Original Content:
The government announced a new policy to boost economic growth...

Generated Summary:
The government announced a new policy to boost economic growth...

---------------------------------------------------

Original Content:
Scientists make significant progress in cancer treatment research...

Generated Summary:
Scientists make significant progress in cancer treatment research...

---------------------------------------------------

Original Content:
AI technology is transforming customer service with faster response times...

Generated Summary:
AI technology is transforming customer service with faster response times...

---------------------------------------------------

Original Content:
Europe experiences record-breaking heatwave with temperatures soaring above 40°C...

Generated Summary:
Europe experiences record-breaking heatwave with temperatures soaring above 40C...

---------------------------------------------------

Original Content:
Researchers unveil AI t

In [ ]:
!pip install rouge_score

import pandas as pd
from rouge_score import rouge_scorer

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=43306d69e3d614895a3c5cf0e966ef0596f3e6c7224530c9fd08c552b5cc8518
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


In [ ]:
def compute_rouge_scores(references, summaries):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = {'rouge1': {'p': 0, 'r': 0, 'f': 0},
              'rouge2': {'p': 0, 'r': 0, 'f': 0},
              'rougeL': {'p': 0, 'r': 0, 'f': 0}} # Changed keys to match the output of scorer.score

    for ref, summ in zip(references, summaries):
        result = scorer.score(ref, summ)
        for metric, score in result.items():
            scores[metric]['p'] += score.precision # Removed .lower() to use keys as returned by scorer.score
            scores[metric]['r'] += score.recall
            scores[metric]['f'] += score.fmeasure

    # Average the scores
    for metric in scores.keys():
        for key in scores[metric].keys():
            scores[metric][key] /= len(references)

    return scores

rouge_scores = compute_rouge_scores(df['News Content in Detail'].tolist(), df['News Content Summary'].tolist())

# Print ROUGE scores
print("ROUGE Scores:")
print(f"ROUGE-1: Precision={rouge_scores['rouge1']['p']}, Recall={rouge_scores['rouge1']['r']}, F1-Score={rouge_scores['rouge1']['f']}")
print(f"ROUGE-2: Precision={rouge_scores['rouge2']['p']}, Recall={rouge_scores['rouge2']['r']}, F1-Score={rouge_scores['rouge2']['f']}")
print(f"ROUGE-L: Precision={rouge_scores['rougeL']['p']}, Recall={rouge_scores['rougeL']['r']}, F1-Score={rouge_scores['rougeL']['f']}")

# Save updated DataFrame back to CSV
df.to_csv('summarized_file.csv', index=False)

ROUGE Scores:
ROUGE-1: Precision=0.9936026936026937, Recall=0.9884297520661157, F1-Score=0.9908863066757804
ROUGE-2: Precision=0.9928451178451178, Recall=0.9872053872053872, F1-Score=0.9898677174218969
ROUGE-L: Precision=0.9936026936026937, Recall=0.9884297520661157, F1-Score=0.9908863066757804
